# The legend

One call puts a legend on the map, and it **derives from the same layer state
the map renders** — glyphs per geometry, sections per sidebar folder, ramps,
bins and categories straight from `color_col`, a stated size row for
`radius_col` — so there is nothing to fall out of step and nothing to register
by hand. Manual entries exist for everything derivation cannot know.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

rng = np.random.default_rng(21)
n = 150
df = pd.DataFrame({
    "lat": 36.03 + rng.normal(0, 0.05, n),
    "lon": -5.45 + rng.normal(0, 0.08, n),
    "reading": np.round(rng.gamma(4, 4, n), 1),
    "volume": rng.integers(10, 500, n),
    "status": rng.choice(["Active", "Idle", "Fault"], n),
})
ring = [[35.96, -5.60], [35.96, -5.48], [36.03, -5.48], [36.03, -5.60]]

m = Map()
m.add_circle_markers(df, name="Reading", layer_group="Points",
                     color_col="reading")
m.add_circle_markers(df, name="Status", layer_group="Points",
                     color_col="status", visible=False)
m.add_circle_markers(df, name="Volume", layer_group="Points",
                     color_col="reading", radius_col="volume", visible=False)
m.add_polygon(ring, name="Zone", layer_group="Areas",
              color="crimson", fill_color="#f5c4ac", fill_opacity=0.4)
m.configure_legend(show=True)
m

What appeared, with no further calls: a section per folder, a ramp row for the
numeric `color_col`, category rows for the string one, a polygon glyph drawn the
way the polygon actually draws (fill inside border), and for `radius_col` a
stated size row — `size ∝ volume (10 – 500)` — never a drawn scale, because
legend pixels are not map pixels at any zoom. Hidden layers render dimmed:
the legend is the map's vocabulary, not just its current screen.

## Bins make classes

In [ ]:
m.add_circle_markers(df, name="Binned", layer_group="Points",
                     color_col="reading", color_bins=[10, 20, 30],
                     visible=False);

## Scope and dimming

`scope='visible'` tracks the screen — unchecking a layer drops its row.
`dim_hidden=False` under the default scope gives the print-style static key.

In [ ]:
m.hide("Status")
m.configure_legend(scope="visible");     # Status's rows drop entirely

In [ ]:
m.configure_legend(scope="all", dim_hidden=True)
m.show("Status");                        # back to the full vocabulary

## Title and position

Eight anchors, shared with the time control. `bottom-left` is the default.

In [ ]:
m.configure_legend(title="Harbour key", position="bottom-right");

## Manual entries: the freedom hatch

`legend_add` writes rows derivation cannot know — resolved into the same spec
shapes as derived rows, so they render identically. `layer=` binds an entry to a
live layer's visibility.

In [ ]:
m.legend_add("Restricted zone", shape="polygon", color="#f00",
             fill_color="#ff000044", group="Areas")
m.legend_add("Threat score", colormap="turbo", vmin=0, vmax=100)
m.legend_add("Sightings", categories=["confirmed", "probable", "ruled out"])
m.legend_add("Live reading", shape="circle", color="#4e79a7", layer="Reading");

## Suppression is persistent

`legend_remove` is a matcher, not a one-shot delete: it keeps suppressing across
every re-derivation, so the row stays gone however the layers change.

In [ ]:
m.legend_remove("Binned");               # by label

## Full manual takeover

`auto=False` stops derivation entirely — the legend is exactly your
`legend_add` entries. `legend_clear()` drops every manual entry and suppression
(display options stay).

In [ ]:
m.configure_legend(auto=False)           # only the four manual rows remain

In [ ]:
m.legend_clear()
m.configure_legend(auto=True);           # back to the derived legend

Exports carry the legend and everything configured here — see **08_export** —
and **09_showcase** puts it in a full scene.